In [86]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

In [97]:
the_dataset = pd.read_csv("./train.csv")
the_dataset


,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,56,0,1,110,226,0,0,132,0,0.0,1,0,7,Absence
629996,629996,54,1,4,128,249,1,2,150,0,0.0,2,0,3,Absence
629997,629997,67,1,4,130,275,0,0,149,0,0.0,1,2,7,Presence
629998,629998,52,1,4,140,199,0,2,157,0,0.0,1,0,6,Presence


In [98]:
the_dataset["ST_interaction"] = the_dataset["ST depression"] * the_dataset["Exercise angina"]
the_dataset = pd.get_dummies(the_dataset, columns=["Heart Disease"], drop_first=True)
the_dataset.rename(columns={"Heart Disease_Presence": "Heart Disease"}, inplace=True)

In [99]:
### Model creation ###


heart_disease = the_dataset["Heart Disease"]
the_dataset.drop(columns=["Heart Disease", "BP", "id", "FBS over 120"], inplace=True)


In [34]:
X_train, X_test, y_train, y_test = train_test_split(the_dataset, heart_disease, test_size=0.2, random_state=42, stratify=heart_disease)

In [100]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns



preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

# Models
model = XGBClassifier(subsample= 1.0, scale_pos_weight= 1, n_estimators= 500, min_child_weight= 5, max_depth= 3, learning_rate= 0.1, gamma= 0.1, colsample_bytree= 1.0)

pipeline = Pipeline(steps=[
    ('scaling', preprocessor),
    ("model", model)
])

pipeline.fit(the_dataset, heart_disease)
predictions = pipeline.predict(X_test)




0.8876352127633146

In [101]:
test_dataset = pd.read_csv("./test.csv")
test_dataset["ST_interaction"] = test_dataset["ST depression"] * test_dataset["Exercise angina"]
test_dataset

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,ST_interaction
0,630000,58,1,3,120,288,0,2,145,1,0.8,2,3,3,0.8
1,630001,55,0,2,120,209,0,0,172,0,0.0,1,0,3,0.0
2,630002,54,1,4,120,268,0,0,150,1,0.0,2,3,7,0.0
3,630003,44,0,3,112,177,0,0,168,0,0.9,1,0,3,0.0
4,630004,43,1,1,138,267,0,0,163,0,1.8,2,0,7,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269995,899995,58,1,2,120,222,0,0,172,0,1.0,1,0,7,0.0
269996,899996,58,1,4,132,289,0,0,172,0,2.8,2,0,3,0.0
269997,899997,63,1,3,108,201,1,0,158,0,0.8,1,0,3,0.0
269998,899998,59,1,4,120,274,0,2,163,0,0.5,1,0,3,0.0


In [102]:
test_dataset2 = test_dataset.copy()
test_dataset2.drop(columns=["id", "BP", "FBS over 120"], inplace =True)

In [103]:
test_dataset2

,Age,Sex,Chest pain type,Cholesterol,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,ST_interaction
0,58,1,3,288,2,145,1,0.8,2,3,3,0.8
1,55,0,2,209,0,172,0,0.0,1,0,3,0.0
2,54,1,4,268,0,150,1,0.0,2,3,7,0.0
3,44,0,3,177,0,168,0,0.9,1,0,3,0.0
4,43,1,1,267,0,163,0,1.8,2,0,7,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
269995,58,1,2,222,0,172,0,1.0,1,0,7,0.0
269996,58,1,4,289,0,172,0,2.8,2,0,3,0.0
269997,63,1,3,201,0,158,0,0.8,1,0,3,0.0
269998,59,1,4,274,2,163,0,0.5,1,0,3,0.0


In [111]:
heart_d = pipeline.predict(test_dataset2)

id = test_dataset[["id"]]

heart = pd.DataFrame(heart_d)
heart

,0
0,1
1,0
2,1
3,0
4,0
...,...
269995,0
269996,1
269997,0
269998,0


In [116]:
submit = pd.concat([id, heart], axis=1)

In [125]:
submit.rename(columns={0: "Heart Disease"}, inplace=True)

In [129]:
submit.to_csv("Submit.csv", index=False)

In [128]:
submit

,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0
...,...,...
269995,899995,0
269996,899996,1
269997,899997,0
269998,899998,0
